In [ ]:
#!/usr/bin/env python3
r"""
Match male-cns Mi1 neurons to FlyWire Mi1 neurons with full-skeleton comparisons.

This script does four things:

1. Fetch Mi1 neurons from:
   - neuPrint dataset ``male-cns:v0.9``
   - FlyWire dataset/materialization ``public`` / ``783``
2. Fetch skeleton nodes for every Mi1 neuron in both datasets.
3. Transform all skeleton nodes into a common template space.
4. For each neuPrint Mi1 neuron ``N``, compare it against every FlyWire Mi1
   neuron ``F`` with the directed score:

       mean_{p in N} [ min_{q in F} dist(p, q) ]

   The FlyWire neuron with the smallest score is selected as the match.

The script writes one output file:

    - neuprint-mi1-to-flywire-v783-directed-mean-nearest-skeleton-matches.csv

Notebook usage
--------------
This file can be imported from a Jupyter notebook and run as a function:

    >>> from pathlib import Path
    >>> from match_mi1_male_cns_to_flywire import run_matching
    >>> result = run_matching(
    ...     output_dir=Path("path/to/workspace")
    ...               / "mi1_male_cns_flywire_match",
    ...     verbose=True,
    ... )
    >>> result["matches"].head()

Notes
-----
This script depends on optional packages that are not part of the minimal local
environment:

    pip install neuprint-python navis fafbseg flybrains pandas scipy

For FlyWire queries you also need a FlyWire/CAVE token configured for ``fafbseg``.
See:
    https://fafbseg-py.readthedocs.io/en/latest/source/tutorials/flywire_setup.html

If the male-cns transform fails with the default source space
``JRCFIB2022Mraw``, try rerunning with ``--neuprint-source-space MANCraw``.
"""

from __future__ import annotations

import argparse
import contextlib
import json
import os
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from neuprint import Client
from scipy.spatial import cKDTree


DEFAULT_NEUPRINT_SERVER = "neuprint.janelia.org"
DEFAULT_NEUPRINT_DATASET = "male-cns:v0.9"
DEFAULT_FLYWIRE_DATASET = "public"
DEFAULT_FLYWIRE_MATERIALIZATION = 783
DEFAULT_CELL_TYPE = "Mi1"
DEFAULT_COMMON_SPACE = "JRCFIB2022M"
DEFAULT_NEUPRINT_SOURCE_SPACE = "JRCFIB2022Mraw"
DEFAULT_FLYWIRE_SOURCE_SPACE = "FLYWIRE"
DEFAULT_MIRROR_FLYWIRE = True
DEFAULT_FETCH_RETRIES = 3

# Paste your neuPrint token here if you want to avoid reading token
# files/environment variables. Leave it as an empty string to keep the
# existing fallback behavior.
DIRECT_NEUPRINT_TOKEN = ""

NEUPRINT_SIDE_MAP = {
    "L": "left",
    "LEFT": "left",
    "LHS": "left",
    "R": "right",
    "RIGHT": "right",
    "RHS": "right",
    "M": "middle",
    "MID": "middle",
    "MIDLINE": "middle",
    "CENTER": "middle",
    "CENTRE": "middle",
}

FLYWIRE_SIDE_MAP = {
    "LEFT": "left",
    "RIGHT": "right",
    "MIDLINE": "middle",
    "CENTER": "middle",
    "CENTRE": "middle",
}


def running_in_notebook() -> bool:
    try:
        from IPython import get_ipython  # noqa: WPS433
    except Exception:
        return False

    shell = get_ipython()
    if shell is None:
        return False
    return shell.__class__.__name__ == "ZMQInteractiveShell"


def script_dir() -> Path:
    try:
        return Path(__file__).resolve().parent
    except NameError:
        return Path.cwd()


SCRIPT_DIR = script_dir()
DEFAULT_OUTPUT_DIR = SCRIPT_DIR.parent / "Processed_Data"
DEFAULT_TOKEN_FILE = SCRIPT_DIR / ".neuprint_token"


@contextlib.contextmanager
def suppress_external_output() -> object:
    with open(os.devnull, "w", encoding="utf-8") as devnull:
        with contextlib.redirect_stdout(devnull), contextlib.redirect_stderr(devnull):
            yield


def print_progress(message: str, *, enabled: bool) -> None:
    if enabled:
        print(message, flush=True)


def print_counter_progress(prefix: str, idx: int, total: int, *, enabled: bool) -> None:
    if not enabled:
        return
    end = "\n" if running_in_notebook() or idx >= total else "\r"
    print(f"{prefix} {idx}/{total}", end=end, flush=True)


def parse_args() -> argparse.Namespace:
    parser = argparse.ArgumentParser(
        description=(
            "Fetch neuprint and FlyWire Mi1 skeletons, transform all skeleton "
            "nodes into a common space, and match each neuprint Mi1 to the "
            "FlyWire Mi1 with the smallest directed mean nearest-node distance."
        )
    )
    parser.add_argument(
        "--cell-type",
        default=DEFAULT_CELL_TYPE,
        help=f"cell type to match (default: {DEFAULT_CELL_TYPE})",
    )
    parser.add_argument(
        "--neuprint-server",
        default=DEFAULT_NEUPRINT_SERVER,
        help=f"neuprint server hostname (default: {DEFAULT_NEUPRINT_SERVER})",
    )
    parser.add_argument(
        "--neuprint-dataset",
        default=DEFAULT_NEUPRINT_DATASET,
        help=f"neuprint dataset name (default: {DEFAULT_NEUPRINT_DATASET})",
    )
    parser.add_argument(
        "--neuprint-source-space",
        default=DEFAULT_NEUPRINT_SOURCE_SPACE,
        help=(
            "navis/flybrains source space for neuprint skeleton coordinates "
            f"(default: {DEFAULT_NEUPRINT_SOURCE_SPACE})"
        ),
    )
    parser.add_argument(
        "--flywire-dataset",
        default=DEFAULT_FLYWIRE_DATASET,
        help=f"FlyWire dataset name (default: {DEFAULT_FLYWIRE_DATASET})",
    )
    parser.add_argument(
        "--flywire-materialization",
        default=str(DEFAULT_FLYWIRE_MATERIALIZATION),
        help=(
            "FlyWire materialization to query for annotations "
            f"(default: {DEFAULT_FLYWIRE_MATERIALIZATION}; also accepts 'v783')"
        ),
    )
    parser.add_argument(
        "--flywire-source-space",
        default=DEFAULT_FLYWIRE_SOURCE_SPACE,
        help=(
            "navis/flybrains source space for FlyWire skeleton coordinates "
            f"(default: {DEFAULT_FLYWIRE_SOURCE_SPACE}; precomputed FlyWire "
            "skeletons are in nanometers, not neuroglancer voxels)"
        ),
    )
    parser.add_argument(
        "--mirror-flywire",
        action=argparse.BooleanOptionalAction,
        default=DEFAULT_MIRROR_FLYWIRE,
        help=(
            "mirror FlyWire left/right before transforming into the common space "
            f"(default: {DEFAULT_MIRROR_FLYWIRE})"
        ),
    )
    parser.add_argument(
        "--common-space",
        default=DEFAULT_COMMON_SPACE,
        help=f"common navis template space (default: {DEFAULT_COMMON_SPACE})",
    )
    parser.add_argument(
        "--output-dir",
        default=str(DEFAULT_OUTPUT_DIR),
        help=f"directory for outputs (default: {DEFAULT_OUTPUT_DIR})",
    )
    parser.add_argument(
        "--token-file",
        default=str(DEFAULT_TOKEN_FILE),
        help=f"neuprint token file path (default: {DEFAULT_TOKEN_FILE})",
    )
    parser.add_argument(
        "--cache-dir",
        default=None,
        help=(
            "shared parent directory for cached skeleton node files "
            "(default: <output-dir>/skeleton_cache)"
        ),
    )
    parser.add_argument(
        "--neuprint-cache-dir",
        default=None,
        help=(
            "directory for neuprint cached skeleton node files "
            "(default: <cache-dir>/neuprint)"
        ),
    )
    parser.add_argument(
        "--flywire-cache-dir",
        default=None,
        help=(
            "directory for FlyWire cached skeleton node files "
            "(default: <cache-dir>/flywire)"
        ),
    )
    parser.add_argument(
        "--fetch-retries",
        type=int,
        default=DEFAULT_FETCH_RETRIES,
        help=f"number of fetch retries per neuron/root (default: {DEFAULT_FETCH_RETRIES})",
    )
    parser.add_argument(
        "--verbose",
        action="store_true",
        help="print extra progress information",
    )
    if running_in_notebook():
        args, unknown = parser.parse_known_args()
        if unknown:
            print(
                "Ignoring notebook/kernel arguments: " + " ".join(unknown),
                file=sys.stderr,
            )
        return args
    return parser.parse_args()


def ensure_optional_dependencies():
    try:
        import navis  # noqa: WPS433
        import flybrains  # noqa: WPS433,F401
        import fafbseg  # noqa: WPS433,F401
        from fafbseg import flywire  # noqa: WPS433
    except ImportError as exc:
        raise RuntimeError(
            "Missing optional dependency required for transforms/FlyWire access. "
            "Please install: neuprint-python navis fafbseg flybrains"
        ) from exc

    return navis, flywire


def load_neuprint_token(token_file: str | Path) -> str:
    direct_token = DIRECT_NEUPRINT_TOKEN.strip()
    if direct_token:
        return direct_token

    token = os.environ.get("NEUPRINT_APPLICATION_CREDENTIALS")
    if token:
        return token

    candidate_paths: list[Path] = []
    if token_file:
        candidate_paths.append(Path(token_file).expanduser())

    candidate_paths.extend(
        [
            SCRIPT_DIR / ".neuprint_token",
            Path.cwd() / ".neuprint_token",
            Path.home() / ".neuprint_token",
        ]
    )

    seen: set[Path] = set()
    unique_candidates: list[Path] = []
    for path in candidate_paths:
        resolved = path.resolve()
        if resolved in seen:
            continue
        seen.add(resolved)
        unique_candidates.append(resolved)

    for token_path in unique_candidates:
        if not token_path.exists():
            continue
        token = token_path.read_text(encoding="utf-8").strip()
        if token:
            return token

    searched = "\n".join(f"  - {p}" for p in unique_candidates)
    raise RuntimeError(
        "No neuprint token found. Set NEUPRINT_APPLICATION_CREDENTIALS or place "
        "the token in one of these locations:\n"
        f"{searched}"
    )


def build_neuprint_client(server: str, dataset: str, token_file: str | Path) -> Client:
    token = load_neuprint_token(token_file)
    return Client(server, dataset=dataset, token=token)


def normalize_side(value, side_map: dict[str, str]) -> str:
    if value is None or (isinstance(value, float) and np.isnan(value)):
        return "unknown"

    text = str(value).strip()
    if not text:
        return "unknown"

    return side_map.get(text.upper(), text.lower())


def parse_flywire_materialization(value: int | str) -> int | str:
    if isinstance(value, int):
        return value

    text = str(value).strip()
    if not text:
        raise ValueError("FlyWire materialization must not be empty.")

    lowered = text.lower()
    if lowered in {"auto", "latest", "live"}:
        return lowered

    if lowered.startswith("v") and lowered[1:].isdigit():
        return int(lowered[1:])

    if lowered.isdigit():
        return int(lowered)

    raise ValueError(
        "Unsupported FlyWire materialization value "
        f"{value!r}. Use e.g. 783, 'v783', 'auto', 'latest', or 'live'."
    )


def flywire_materialization_label(materialization: int | str) -> str:
    if isinstance(materialization, int):
        return f"v{materialization}"
    return str(materialization)


class EmptySkeletonError(RuntimeError):
    """Raised when a skeleton fetch succeeds but contains no usable nodes."""


def sanitize_for_path(value: object) -> str:
    text = str(value).strip()
    safe = [
        character if (character.isalnum() or character in {"-", "_", "."}) else "_"
        for character in text
    ]
    slug = "".join(safe).strip("_")
    return slug or "value"


def coerce_single_int(value: object, *, label: str) -> int:
    flattened = np.asarray(value).reshape(-1)
    if flattened.size != 1:
        raise ValueError(
            f"{label} must contain exactly one value, got shape {flattened.shape}."
        )
    return int(flattened[0])


def validate_common_xyz(xyz: np.ndarray, *, label: str) -> np.ndarray:
    validated = np.asarray(xyz, dtype=float)
    if validated.ndim != 2 or validated.shape[1] != 3:
        raise ValueError(
            f"{label} must have shape (n, 3), got {validated.shape!r}."
        )
    if len(validated) == 0:
        raise EmptySkeletonError(f"{label} contains no nodes.")
    if not np.isfinite(validated).all():
        raise ValueError(f"{label} contains NaN or inf coordinates.")
    return validated


def save_cached_skeleton_nodes(
    path: Path,
    xyz: np.ndarray,
    *,
    metadata: dict[str, object],
) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    payload: dict[str, np.ndarray] = {"xyz": np.asarray(xyz, dtype=np.float32)}
    for key, value in metadata.items():
        payload[key] = np.asarray(value)
    np.savez_compressed(path, **payload)


def load_cached_skeleton_nodes(path: Path) -> tuple[np.ndarray, dict[str, object]]:
    with np.load(path, allow_pickle=False) as cached:
        if "xyz" not in cached.files:
            raise ValueError(f"Cached skeleton file {path} does not contain 'xyz'.")
        xyz = validate_common_xyz(cached["xyz"], label=str(path))
        metadata: dict[str, object] = {}
        for key in cached.files:
            if key == "xyz":
                continue
            value = cached[key]
            if getattr(value, "ndim", 0) == 0:
                metadata[key] = value.item()
            elif getattr(value, "size", 0) == 1:
                metadata[key] = value.reshape(()).item()
            else:
                metadata[key] = value.tolist()
    return xyz, metadata


def make_neuprint_cache_dir(
    base_cache_dir: Path,
    *,
    dataset: str,
    cell_type: str,
    requested_source_space: str,
    common_space: str,
) -> Path:
    return (
        base_cache_dir
        / sanitize_for_path(dataset)
        / sanitize_for_path(cell_type)
        / f"{sanitize_for_path(requested_source_space)}_to_{sanitize_for_path(common_space)}"
    )


def make_flywire_cache_dir(
    base_cache_dir: Path,
    *,
    dataset: str,
    materialization_label: str,
    cell_type: str,
    source_space: str,
    mirror_flywire: bool,
    common_space: str,
) -> Path:
    mirror_label = "mirrored" if mirror_flywire else "unmirrored"
    return (
        base_cache_dir
        / sanitize_for_path(dataset)
        / sanitize_for_path(materialization_label)
        / sanitize_for_path(cell_type)
        / mirror_label
        / f"{sanitize_for_path(source_space)}_to_{sanitize_for_path(common_space)}"
    )


def resolve_cache_roots(
    *,
    output_dir: Path,
    cache_dir: str | Path | None,
    neuprint_cache_dir: str | Path | None,
    flywire_cache_dir: str | Path | None,
) -> tuple[Path, Path, Path]:
    shared_cache_root = Path(cache_dir) if cache_dir else output_dir / "skeleton_cache"
    resolved_neuprint_cache_dir = (
        Path(neuprint_cache_dir)
        if neuprint_cache_dir
        else shared_cache_root / "neuprint"
    )
    resolved_flywire_cache_dir = (
        Path(flywire_cache_dir)
        if flywire_cache_dir
        else shared_cache_root / "flywire"
    )
    return shared_cache_root, resolved_neuprint_cache_dir, resolved_flywire_cache_dir


def initialize_fetch_stats(*, id_label: str, requested_count: int) -> dict[str, object]:
    return {
        "id_label": id_label,
        "requested_count": int(requested_count),
        "cache_hit_count": 0,
        "fetched_count": 0,
        "retried_success_count": 0,
        "cache_invalid_ids": [],
        "empty_ids": [],
        "failed_ids": [],
        "issue_rows": [],
    }


def finalize_fetch_stats(stats: dict[str, object], *, success_count: int) -> dict[str, object]:
    stats["success_count"] = int(success_count)
    stats["cache_invalid_count"] = len(stats["cache_invalid_ids"])
    stats["empty_count"] = len(stats["empty_ids"])
    stats["failed_count"] = len(stats["failed_ids"])
    return stats


def make_issue_df(stats: dict[str, object]) -> pd.DataFrame:
    id_label = str(stats["id_label"])
    rows = list(stats["issue_rows"])
    columns = [id_label, "issue_type", "attempts", "error"]
    if not rows:
        return pd.DataFrame(columns=columns)
    return pd.DataFrame(rows, columns=columns)


def print_fetch_summary(prefix: str, stats: dict[str, object]) -> None:
    print(
        f"{prefix}: requested={stats['requested_count']}, "
        f"success={stats['success_count']}, "
        f"cache_hits={stats['cache_hit_count']}, "
        f"fetched={stats['fetched_count']}, "
        f"retried_successes={stats['retried_success_count']}, "
        f"invalid_cache_refetched={stats['cache_invalid_count']}, "
        f"empty={stats['empty_count']}, "
        f"failed_after_retries={stats['failed_count']}",
        flush=True,
    )


def build_neuprint_summary_row(
    row,
    *,
    used_source_space: str,
    common_space: str,
    common_xyz: np.ndarray,
) -> dict[str, object]:
    return {
        "body_id": int(row.body_id),
        "type": row.type,
        "instance": row.instance,
        "side_raw": row.side_raw,
        "side": row.side,
        "dataset": row.dataset,
        "source_space": used_source_space,
        "common_space": common_space,
        "n_skeleton_nodes": int(len(common_xyz)),
    }


def build_flywire_summary_row(
    ann_lookup: pd.DataFrame,
    *,
    root_id: int,
    cell_type: str,
    annotation_dataset: str,
    materialization: int | str,
    common_space: str,
    common_xyz: np.ndarray,
    source_space: str = DEFAULT_FLYWIRE_SOURCE_SPACE,
) -> dict[str, object]:
    row: dict[str, object] = {
        "root_id": int(root_id),
        "type": cell_type,
        "dataset": f"{annotation_dataset}:{materialization}",
        "source_space": source_space,
        "common_space": common_space,
        "n_skeleton_nodes": int(len(common_xyz)),
    }
    if root_id in ann_lookup.index:
        ann_row = ann_lookup.loc[root_id]
        for column in [
            "side_raw",
            "side",
            "cell_type",
            "hemibrain_type",
            "supertype",
            "flow",
            "super_class",
            "cell_class",
        ]:
            if column in ann_row.index:
                row[column] = ann_row[column]
    return row


def normalize_flywire_skeletons(skeletons) -> list[object]:
    if skeletons is None:
        return []
    skeletons_class_name = type(skeletons).__name__.lower()
    if skeletons_class_name.endswith("neuronlist"):
        return list(skeletons)
    if hasattr(skeletons, "nodes"):
        return [skeletons]
    return list(skeletons)


def fetch_neuprint_mi1_metadata(
    client: Client,
    cell_type: str,
    *,
    body_ids: list[int] | None = None,
) -> pd.DataFrame:
    quoted = json.dumps(cell_type)
    where_clauses = [f"n.type = {quoted}"]
    if body_ids:
        body_ids_expr = "[" + ", ".join(str(int(body_id)) for body_id in body_ids) + "]"
        where_clauses.append(f"n.bodyId IN {body_ids_expr}")
    q = f"""
        MATCH (n:Neuron)
        WHERE {' AND '.join(where_clauses)}
        RETURN n.bodyId AS body_id,
               n.type AS type,
               n.instance AS instance,
               n.somaSide AS side_raw
        ORDER BY body_id
    """
    df = client.fetch_custom(q)

    if df.empty:
        raise RuntimeError(
            f"No neuprint neurons found with n.type == {cell_type!r} in "
            f"dataset {client.dataset!r}."
        )

    df["dataset"] = client.dataset
    df["side"] = df["side_raw"].apply(
        lambda v: normalize_side(v, NEUPRINT_SIDE_MAP)
    )
    return df.reset_index(drop=True)


def make_neuprint_source_space_candidates(
    requested_source_space: str,
    preferred_source_space: str | None = None,
) -> list[str]:
    candidates: list[str] = []
    if preferred_source_space:
        candidates.append(preferred_source_space)
    candidates.append(requested_source_space)
    if requested_source_space == DEFAULT_NEUPRINT_SOURCE_SPACE:
        candidates.append("MANCraw")
    return list(dict.fromkeys(candidates))


def transform_xyz_with_candidate_sources(
    navis,
    xyz: np.ndarray,
    source_candidates: list[str],
    target_space: str,
) -> tuple[np.ndarray, str]:
    errors: list[str] = []
    for source_space in source_candidates:
        try:
            with suppress_external_output():
                transformed = np.asarray(
                    navis.xform_brain(xyz, source=source_space, target=target_space),
                    dtype=float,
                )
            return validate_common_xyz(
                transformed,
                label=f"transformed coordinates in {target_space}",
            ), source_space
        except Exception as exc:  # pragma: no cover - depends on transforms/data
            errors.append(f"{source_space}: {exc}")

    raise RuntimeError(
        "Failed to transform coordinates. Tried source spaces: "
        + "; ".join(errors)
    )


def mirror_flywire_xyz(navis, xyz: np.ndarray, *, source_space: str) -> np.ndarray:
    mirrored = navis.mirror_brain(xyz, template=source_space)
    return validate_common_xyz(
        mirrored,
        label=f"FlyWire mirrored coordinates in {source_space}",
    )


def fetch_neuprint_mi1_skeleton_nodes(
    client: Client,
    navis,
    cell_type: str,
    requested_source_space: str,
    common_space: str,
    *,
    cache_dir: str | Path,
    fetch_retries: int = DEFAULT_FETCH_RETRIES,
    body_ids: list[int] | None = None,
    verbose: bool = False,
) -> tuple[pd.DataFrame, dict[int, np.ndarray], dict[str, object]]:
    meta_df = fetch_neuprint_mi1_metadata(client, cell_type, body_ids=body_ids)
    fetch_retries = max(1, int(fetch_retries))
    cache_dir = make_neuprint_cache_dir(
        Path(cache_dir),
        dataset=client.dataset,
        cell_type=cell_type,
        requested_source_space=requested_source_space,
        common_space=common_space,
    )
    nodes_by_body: dict[int, np.ndarray] = {}
    summary_rows: list[dict[str, object]] = []
    stats = initialize_fetch_stats(id_label="body_id", requested_count=len(meta_df))
    resolved_source_space: str | None = None

    total = len(meta_df)
    for idx, row in enumerate(meta_df.itertuples(index=False), start=1):
        body_id = int(row.body_id)
        print_counter_progress(
            "  [neuprint] skeleton",
            idx,
            total,
            enabled=True,
        )

        cache_path = cache_dir / f"body_{body_id}.npz"
        if cache_path.exists():
            try:
                common_xyz, cached_meta = load_cached_skeleton_nodes(cache_path)
                used_source_space = str(
                    cached_meta.get("source_space") or requested_source_space
                )
                nodes_by_body[body_id] = common_xyz
                summary_rows.append(
                    build_neuprint_summary_row(
                        row,
                        used_source_space=used_source_space,
                        common_space=common_space,
                        common_xyz=common_xyz,
                    )
                )
                resolved_source_space = used_source_space
                stats["cache_hit_count"] += 1
                continue
            except Exception as exc:
                stats["cache_invalid_ids"].append(body_id)
                print_progress(
                    f"    [neuprint] invalid cache for body {body_id}; refetching ({exc})",
                    enabled=verbose,
                )

        last_error: Exception | None = None
        succeeded = False
        for attempt in range(1, fetch_retries + 1):
            try:
                with suppress_external_output():
                    skel_df = client.fetch_skeleton(body_id, heal=True, format="pandas")

                if skel_df is None or len(skel_df) == 0:
                    raise EmptySkeletonError("empty skeleton dataframe")
                if not {"x", "y", "z"}.issubset(skel_df.columns):
                    raise ValueError("missing x/y/z columns in neuprint skeleton")

                xyz = validate_common_xyz(
                    skel_df[["x", "y", "z"]].to_numpy(dtype=float),
                    label=f"neuprint body {body_id} source skeleton",
                )
                source_candidates = make_neuprint_source_space_candidates(
                    requested_source_space=requested_source_space,
                    preferred_source_space=resolved_source_space,
                )
                common_xyz, used_source_space = transform_xyz_with_candidate_sources(
                    navis=navis,
                    xyz=xyz,
                    source_candidates=source_candidates,
                    target_space=common_space,
                )
                save_cached_skeleton_nodes(
                    cache_path,
                    common_xyz,
                    metadata={
                        "body_id": body_id,
                        "source_space": used_source_space,
                        "common_space": common_space,
                        "dataset": row.dataset,
                    },
                )
                nodes_by_body[body_id] = common_xyz
                summary_rows.append(
                    build_neuprint_summary_row(
                        row,
                        used_source_space=used_source_space,
                        common_space=common_space,
                        common_xyz=common_xyz,
                    )
                )
                resolved_source_space = used_source_space
                stats["fetched_count"] += 1
                if attempt > 1:
                    stats["retried_success_count"] += 1
                succeeded = True
                break
            except EmptySkeletonError as exc:
                stats["empty_ids"].append(body_id)
                stats["issue_rows"].append(
                    {
                        "body_id": body_id,
                        "issue_type": "empty_skeleton",
                        "attempts": attempt,
                        "error": str(exc),
                    }
                )
                succeeded = True
                break
            except Exception as exc:
                last_error = exc
                if attempt < fetch_retries:
                    print_progress(
                        f"    [neuprint] retry {attempt + 1}/{fetch_retries} for body {body_id}",
                        enabled=verbose,
                    )

        if not succeeded:
            stats["failed_ids"].append(body_id)
            stats["issue_rows"].append(
                {
                    "body_id": body_id,
                    "issue_type": "failed_after_retries",
                    "attempts": fetch_retries,
                    "error": str(last_error) if last_error else "unknown error",
                }
            )
            continue

    if not nodes_by_body:
        raise RuntimeError("No neuprint skeleton nodes were fetched.")

    summary_df = pd.DataFrame(summary_rows).sort_values("body_id").reset_index(drop=True)
    return (
        summary_df,
        nodes_by_body,
        finalize_fetch_stats(stats, success_count=len(nodes_by_body)),
    )


def fetch_flywire_mi1_skeleton_nodes(
    navis,
    flywire,
    cell_type: str,
    annotation_dataset: str,
    materialization: int | str,
    flywire_source_space: str,
    common_space: str,
    *,
    cache_dir: str | Path,
    fetch_retries: int = DEFAULT_FETCH_RETRIES,
    mirror_flywire: bool = DEFAULT_MIRROR_FLYWIRE,
    verbose: bool = False,
) -> tuple[pd.DataFrame, dict[int, np.ndarray], dict[str, object]]:
    criteria = flywire.NeuronCriteria(
        type=cell_type,
        regex=False,
        dataset=annotation_dataset,
        materialization=materialization,
        verbose=False,
    )
    with suppress_external_output():
        ann = flywire.search_annotations(
            criteria,
            dataset=annotation_dataset,
            materialization=materialization,
            verbose=False,
        )

    if ann.empty:
        raise RuntimeError(
            f"No FlyWire annotations found for type={cell_type!r} in "
            f"dataset={annotation_dataset!r}, materialization={materialization!r}."
        )

    roots = ann["root_id"].dropna().astype(np.int64).drop_duplicates().tolist()
    if not roots:
        raise RuntimeError("No FlyWire root IDs found in annotation table.")

    fetch_retries = max(1, int(fetch_retries))
    if not isinstance(materialization, int):
        raise RuntimeError(
            "FlyWire skeleton fetching requires an integer materialization "
            f"(e.g. 783), got {materialization!r}."
        )

    materialization_label = flywire_materialization_label(materialization)
    cache_dir = make_flywire_cache_dir(
        Path(cache_dir),
        dataset=annotation_dataset,
        materialization_label=materialization_label,
        cell_type=cell_type,
        source_space=flywire_source_space,
        mirror_flywire=mirror_flywire,
        common_space=common_space,
    )

    nodes_by_root: dict[int, np.ndarray] = {}
    summary_rows: list[dict[str, object]] = []
    stats = initialize_fetch_stats(id_label="root_id", requested_count=len(roots))

    ann_keep = ann.copy()
    ann_keep["side_raw"] = ann_keep.get("side", pd.Series(index=ann_keep.index))
    ann_keep["side"] = ann_keep["side_raw"].apply(
        lambda v: normalize_side(v, FLYWIRE_SIDE_MAP)
    )
    ann_lookup = ann_keep.drop_duplicates("root_id").set_index("root_id")

    total = len(roots)
    for idx, requested_root_id in enumerate(roots, start=1):
        print_counter_progress(
            "  [flywire] skeleton",
            idx,
            total,
            enabled=True,
        )
        root_id = int(requested_root_id)
        cache_path = cache_dir / f"root_{root_id}.npz"
        if cache_path.exists():
            try:
                common_xyz, cached_meta = load_cached_skeleton_nodes(cache_path)
                nodes_by_root[root_id] = common_xyz
                summary_rows.append(
                    build_flywire_summary_row(
                        ann_lookup,
                        root_id=root_id,
                        cell_type=cell_type,
                        annotation_dataset=annotation_dataset,
                        materialization=materialization,
                        common_space=common_space,
                        common_xyz=common_xyz,
                        source_space=str(
                            cached_meta.get("source_space") or flywire_source_space
                        ),
                    )
                )
                stats["cache_hit_count"] += 1
                continue
            except Exception as exc:
                stats["cache_invalid_ids"].append(root_id)
                print_progress(
                    f"    [flywire] invalid cache for root {root_id}; refetching ({exc})",
                    enabled=verbose,
                )

        last_error: Exception | None = None
        succeeded = False
        for attempt in range(1, fetch_retries + 1):
            try:
                with suppress_external_output():
                    skeletons = flywire.get_skeletons(
                        [root_id],
                        dataset=materialization,
                        omit_failures=False,
                        progress=False,
                    )
                skeleton_iter = normalize_flywire_skeletons(skeletons)
                if not skeleton_iter:
                    raise EmptySkeletonError("empty FlyWire skeleton response")

                matched_skeleton = None
                for skel in skeleton_iter:
                    candidate_root_id = coerce_single_int(
                        getattr(skel, "id"),
                        label="FlyWire skeleton id",
                    )
                    if candidate_root_id == root_id:
                        matched_skeleton = skel
                        break
                if matched_skeleton is None:
                    raise ValueError(
                        f"FlyWire root {root_id} was not present in the returned skeletons."
                    )

                nodes = getattr(matched_skeleton, "nodes", None)
                if nodes is None or len(nodes) == 0:
                    raise EmptySkeletonError("FlyWire skeleton has no nodes")
                if not {"x", "y", "z"}.issubset(nodes.columns):
                    raise ValueError("missing x/y/z columns in FlyWire skeleton")

                xyz = validate_common_xyz(
                    nodes[["x", "y", "z"]].to_numpy(dtype=float),
                    label=f"FlyWire root {root_id} source skeleton",
                )
                if mirror_flywire:
                    xyz = mirror_flywire_xyz(
                        navis,
                        xyz,
                        source_space=flywire_source_space,
                    )
                with suppress_external_output():
                    common_xyz = np.asarray(
                        navis.xform_brain(
                            xyz,
                            source=flywire_source_space,
                            target=common_space,
                        ),
                        dtype=float,
                    )
                common_xyz = validate_common_xyz(
                    common_xyz,
                    label=f"FlyWire root {root_id} transformed skeleton",
                )
                save_cached_skeleton_nodes(
                    cache_path,
                    common_xyz,
                    metadata={
                        "root_id": root_id,
                        "source_space": flywire_source_space,
                        "mirror_flywire": mirror_flywire,
                        "common_space": common_space,
                        "dataset": f"{annotation_dataset}:{materialization}",
                    },
                )
                nodes_by_root[root_id] = common_xyz
                summary_rows.append(
                    build_flywire_summary_row(
                        ann_lookup,
                        root_id=root_id,
                        cell_type=cell_type,
                        annotation_dataset=annotation_dataset,
                        materialization=materialization,
                        common_space=common_space,
                        common_xyz=common_xyz,
                        source_space=flywire_source_space,
                    )
                )
                stats["fetched_count"] += 1
                if attempt > 1:
                    stats["retried_success_count"] += 1
                succeeded = True
                break
            except EmptySkeletonError as exc:
                stats["empty_ids"].append(root_id)
                stats["issue_rows"].append(
                    {
                        "root_id": root_id,
                        "issue_type": "empty_skeleton",
                        "attempts": attempt,
                        "error": str(exc),
                    }
                )
                succeeded = True
                break
            except Exception as exc:
                last_error = exc
                if attempt < fetch_retries:
                    print_progress(
                        f"    [flywire] retry {attempt + 1}/{fetch_retries} for root {root_id}",
                        enabled=verbose,
                    )

        if not succeeded:
            stats["failed_ids"].append(root_id)
            stats["issue_rows"].append(
                {
                    "root_id": root_id,
                    "issue_type": "failed_after_retries",
                    "attempts": fetch_retries,
                    "error": str(last_error) if last_error else "unknown error",
                }
            )

    if not nodes_by_root:
        raise RuntimeError("No FlyWire skeleton nodes were fetched.")

    summary_df = pd.DataFrame(summary_rows).sort_values("root_id").reset_index(drop=True)
    return (
        summary_df,
        nodes_by_root,
        finalize_fetch_stats(stats, success_count=len(nodes_by_root)),
    )


def match_by_directed_mean_nearest_skeleton_distance(
    neuprint_summary_df: pd.DataFrame,
    neuprint_nodes_by_body: dict[int, np.ndarray],
    flywire_summary_df: pd.DataFrame,
    flywire_nodes_by_root: dict[int, np.ndarray],
    *,
    verbose: bool = False,
) -> pd.DataFrame:
    if not flywire_nodes_by_root:
        raise RuntimeError("No FlyWire skeleton nodes available for matching.")

    flywire_roots = sorted(flywire_nodes_by_root)
    flywire_trees = {
        root_id: cKDTree(nodes)
        for root_id, nodes in flywire_nodes_by_root.items()
    }

    match_rows: list[dict[str, object]] = []
    total = len(neuprint_summary_df)
    for idx, row in enumerate(neuprint_summary_df.itertuples(index=False), start=1):
        body_id = int(row.body_id)
        neuprint_nodes = neuprint_nodes_by_body[body_id]
        print_counter_progress(
            "  [match] neuprint skeleton",
            idx,
            total,
            enabled=True,
        )

        best_root_id: int | None = None
        best_mean = np.inf
        best_min = np.nan
        best_max = np.nan
        best_std = np.nan

        for root_id in flywire_roots:
            distances, _ = flywire_trees[root_id].query(neuprint_nodes, k=1)
            mean_distance = float(np.mean(distances))
            if mean_distance < best_mean:
                best_root_id = int(root_id)
                best_mean = mean_distance
                best_min = float(np.min(distances))
                best_max = float(np.max(distances))
                best_std = float(np.std(distances))

        row_dict = row._asdict()
        row_dict.update(
            {
                "flywire_root_id": best_root_id,
                "flywire_candidate_count": int(len(flywire_roots)),
                "mean_min_distance_in_common_space": best_mean,
                "min_min_distance_in_common_space": best_min,
                "max_min_distance_in_common_space": best_max,
                "std_min_distance_in_common_space": best_std,
                "match_status": "matched_by_directed_mean_nearest_skeleton_distance",
            }
        )
        match_rows.append(row_dict)

    matches = pd.DataFrame(match_rows)
    flywire_meta = flywire_summary_df.add_prefix("flywire_")
    matches = matches.merge(
        flywire_meta,
        left_on="flywire_root_id",
        right_on="flywire_root_id",
        how="left",
    )

    matches = matches.rename(
        columns={
            "body_id": "neuprint_body_id",
            "type": "neuprint_type",
            "instance": "neuprint_instance",
            "side_raw": "neuprint_side_raw",
            "side": "neuprint_side",
            "dataset": "neuprint_dataset",
            "source_space": "neuprint_source_space",
            "common_space": "common_space",
            "n_skeleton_nodes": "neuprint_n_skeleton_nodes",
        }
    )
    return matches.sort_values("neuprint_body_id").reset_index(drop=True)


def score_single_neuprint_skeleton_against_all_flywire(
    neuprint_row: pd.Series,
    neuprint_nodes: np.ndarray,
    flywire_summary_df: pd.DataFrame,
    flywire_nodes_by_root: dict[int, np.ndarray],
    *,
    top_k: int | None = None,
) -> pd.DataFrame:
    flywire_roots = sorted(flywire_nodes_by_root)
    flywire_trees = {
        root_id: cKDTree(nodes)
        for root_id, nodes in flywire_nodes_by_root.items()
    }

    candidate_rows: list[dict[str, object]] = []
    for root_id in flywire_roots:
        distances, _ = flywire_trees[root_id].query(neuprint_nodes, k=1)
        candidate_rows.append(
            {
                "neuprint_body_id": int(neuprint_row["body_id"]),
                "neuprint_type": neuprint_row["type"],
                "neuprint_instance": neuprint_row["instance"],
                "neuprint_side_raw": neuprint_row["side_raw"],
                "neuprint_side": neuprint_row["side"],
                "neuprint_dataset": neuprint_row["dataset"],
                "neuprint_source_space": neuprint_row["source_space"],
                "common_space": neuprint_row["common_space"],
                "neuprint_n_skeleton_nodes": int(neuprint_row["n_skeleton_nodes"]),
                "flywire_root_id": int(root_id),
                "mean_min_distance_in_common_space": float(np.mean(distances)),
                "min_min_distance_in_common_space": float(np.min(distances)),
                "max_min_distance_in_common_space": float(np.max(distances)),
                "std_min_distance_in_common_space": float(np.std(distances)),
            }
        )

    candidates = pd.DataFrame(candidate_rows)
    flywire_meta = flywire_summary_df.add_prefix("flywire_")
    candidates = candidates.merge(
        flywire_meta,
        left_on="flywire_root_id",
        right_on="flywire_root_id",
        how="left",
    )
    candidates = candidates.sort_values(
        ["mean_min_distance_in_common_space", "min_min_distance_in_common_space", "flywire_root_id"]
    ).reset_index(drop=True)
    candidates["rank"] = np.arange(1, len(candidates) + 1, dtype=int)

    if top_k is not None:
        candidates = candidates.head(int(top_k)).reset_index(drop=True)
    return candidates


def write_csv(df: pd.DataFrame, path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(path, index=False)


def write_nodes_npz(nodes_by_id: dict[int, np.ndarray], path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    arrays = {
        str(node_id): np.asarray(xyz, dtype=np.float32)
        for node_id, xyz in nodes_by_id.items()
    }
    np.savez_compressed(path, **arrays)


def run_matching(
    *,
    cell_type: str = DEFAULT_CELL_TYPE,
    neuprint_server: str = DEFAULT_NEUPRINT_SERVER,
    neuprint_dataset: str = DEFAULT_NEUPRINT_DATASET,
    neuprint_source_space: str = DEFAULT_NEUPRINT_SOURCE_SPACE,
    flywire_dataset: str = DEFAULT_FLYWIRE_DATASET,
    flywire_materialization: int | str = DEFAULT_FLYWIRE_MATERIALIZATION,
    flywire_source_space: str = DEFAULT_FLYWIRE_SOURCE_SPACE,
    mirror_flywire: bool = DEFAULT_MIRROR_FLYWIRE,
    common_space: str = DEFAULT_COMMON_SPACE,
    output_dir: str | Path | None = None,
    cache_dir: str | Path | None = None,
    neuprint_cache_dir: str | Path | None = None,
    flywire_cache_dir: str | Path | None = None,
    token_file: str | Path = DEFAULT_TOKEN_FILE,
    fetch_retries: int = DEFAULT_FETCH_RETRIES,
    write_outputs: bool = True,
    verbose: bool = False,
) -> dict[str, object]:
    output_dir = Path(output_dir) if output_dir else DEFAULT_OUTPUT_DIR
    cache_dir, neuprint_cache_dir, flywire_cache_dir = resolve_cache_roots(
        output_dir=output_dir,
        cache_dir=cache_dir,
        neuprint_cache_dir=neuprint_cache_dir,
        flywire_cache_dir=flywire_cache_dir,
    )
    neuprint_fetch_stats: dict[str, object] | None = None
    flywire_fetch_stats: dict[str, object] | None = None

    try:
        parsed_flywire_materialization = parse_flywire_materialization(
            flywire_materialization
        )
        materialization_label = flywire_materialization_label(
            parsed_flywire_materialization
        )
        navis, flywire = ensure_optional_dependencies()
        client = build_neuprint_client(
            server=neuprint_server,
            dataset=neuprint_dataset,
            token_file=token_file,
        )

        print_progress(
            f"[1/4] Fetching neuprint {cell_type} skeletons from {neuprint_dataset}",
            enabled=True,
        )
        (
            neuprint_summary_df,
            neuprint_nodes_by_body,
            neuprint_fetch_stats,
        ) = fetch_neuprint_mi1_skeleton_nodes(
            client=client,
            navis=navis,
            cell_type=cell_type,
            requested_source_space=neuprint_source_space,
            common_space=common_space,
            cache_dir=neuprint_cache_dir,
            fetch_retries=fetch_retries,
            verbose=verbose,
        )

        print_progress(
            f"[2/4] Fetching FlyWire {cell_type} skeletons from "
            f"{flywire_dataset}:{parsed_flywire_materialization}",
            enabled=True,
        )
        (
            flywire_summary_df,
            flywire_nodes_by_root,
            flywire_fetch_stats,
        ) = fetch_flywire_mi1_skeleton_nodes(
            navis=navis,
            flywire=flywire,
            cell_type=cell_type,
            annotation_dataset=flywire_dataset,
            materialization=parsed_flywire_materialization,
            flywire_source_space=flywire_source_space,
            common_space=common_space,
            cache_dir=flywire_cache_dir,
            fetch_retries=fetch_retries,
            mirror_flywire=mirror_flywire,
            verbose=verbose,
        )

        print_progress(
            "[3/4] Computing directed mean nearest-skeleton distances "
            "(neuprint -> FlyWire)",
            enabled=True,
        )
        matches = match_by_directed_mean_nearest_skeleton_distance(
            neuprint_summary_df=neuprint_summary_df,
            neuprint_nodes_by_body=neuprint_nodes_by_body,
            flywire_summary_df=flywire_summary_df,
            flywire_nodes_by_root=flywire_nodes_by_root,
            verbose=verbose,
        )
    except Exception as exc:
        if neuprint_fetch_stats is not None:
            print_fetch_summary("neuprint fetch summary", neuprint_fetch_stats)
        if flywire_fetch_stats is not None:
            print_fetch_summary("FlyWire fetch summary", flywire_fetch_stats)
        raise RuntimeError(f"Matching failed: {exc}") from exc

    neuprint_summary_out = output_dir / "neuprint-mi1-skeleton-summary.csv"
    neuprint_nodes_out = output_dir / "neuprint-mi1-skeleton-nodes-common-space.npz"
    neuprint_issues_out = output_dir / "neuprint-mi1-skeleton-fetch-issues.csv"
    flywire_summary_out = output_dir / f"flywire-{materialization_label}-mi1-skeleton-summary.csv"
    flywire_nodes_out = output_dir / f"flywire-{materialization_label}-mi1-skeleton-nodes-common-space.npz"
    flywire_issues_out = output_dir / f"flywire-{materialization_label}-mi1-skeleton-fetch-issues.csv"
    matches_out = output_dir / (
        f"neuprint-mi1-to-flywire-{materialization_label}-"
        "directed-mean-nearest-skeleton-matches.csv"
    )
    neuprint_issue_df = make_issue_df(neuprint_fetch_stats)
    flywire_issue_df = make_issue_df(flywire_fetch_stats)

    if write_outputs:
        print_progress("[4/4] Writing output file", enabled=True)
        write_csv(matches, matches_out)

        print(f"Wrote {len(matches):,} directed skeleton matches to {matches_out}")
        print(f"Shared skeleton cache root: {cache_dir}")
        print(f"neuprint skeleton cache directory: {neuprint_cache_dir}")
        print(f"FlyWire skeleton cache directory: {flywire_cache_dir}")
        print(f"FlyWire source space: {flywire_source_space}")
        print(f"FlyWire mirrored before transform: {mirror_flywire}")
        print_fetch_summary("neuprint fetch summary", neuprint_fetch_stats)
        print_fetch_summary("FlyWire fetch summary", flywire_fetch_stats)
        print(
            "Matching rule: for each neuprint Mi1 N and FlyWire Mi1 F, "
            "score(N, F) = mean over all nodes p in N of min distance from "
            f"p to any node in F, measured in {common_space}."
        )

    return {
        "neuprint_summary": neuprint_summary_df.sort_values("body_id").reset_index(drop=True),
        "neuprint_nodes": neuprint_nodes_by_body,
        "flywire_summary": flywire_summary_df.sort_values("root_id").reset_index(drop=True),
        "flywire_nodes": flywire_nodes_by_root,
        "matches": matches,
        "output_dir": output_dir,
        "cache_dir": cache_dir,
        "neuprint_cache_dir": neuprint_cache_dir,
        "flywire_cache_dir": flywire_cache_dir,
        "flywire_source_space": flywire_source_space,
        "mirror_flywire": mirror_flywire,
        "neuprint_summary_output": neuprint_summary_out,
        "neuprint_nodes_output": neuprint_nodes_out,
        "neuprint_issues_output": neuprint_issues_out,
        "flywire_summary_output": flywire_summary_out,
        "flywire_nodes_output": flywire_nodes_out,
        "flywire_issues_output": flywire_issues_out,
        "matches_output": matches_out,
        "neuprint_fetch_stats": neuprint_fetch_stats,
        "flywire_fetch_stats": flywire_fetch_stats,
    }


def run_random_single_match(
    *,
    cell_type: str = DEFAULT_CELL_TYPE,
    neuprint_server: str = DEFAULT_NEUPRINT_SERVER,
    neuprint_dataset: str = DEFAULT_NEUPRINT_DATASET,
    neuprint_source_space: str = DEFAULT_NEUPRINT_SOURCE_SPACE,
    flywire_dataset: str = DEFAULT_FLYWIRE_DATASET,
    flywire_materialization: int | str = DEFAULT_FLYWIRE_MATERIALIZATION,
    flywire_source_space: str = DEFAULT_FLYWIRE_SOURCE_SPACE,
    mirror_flywire: bool = DEFAULT_MIRROR_FLYWIRE,
    common_space: str = DEFAULT_COMMON_SPACE,
    output_dir: str | Path | None = None,
    cache_dir: str | Path | None = None,
    neuprint_cache_dir: str | Path | None = None,
    flywire_cache_dir: str | Path | None = None,
    token_file: str | Path = DEFAULT_TOKEN_FILE,
    random_seed: int | None = None,
    top_k: int = 20,
    fetch_retries: int = DEFAULT_FETCH_RETRIES,
    write_outputs: bool = True,
    verbose: bool = False,
) -> dict[str, object]:
    output_dir = Path(output_dir) if output_dir else DEFAULT_OUTPUT_DIR
    cache_dir, neuprint_cache_dir, flywire_cache_dir = resolve_cache_roots(
        output_dir=output_dir,
        cache_dir=cache_dir,
        neuprint_cache_dir=neuprint_cache_dir,
        flywire_cache_dir=flywire_cache_dir,
    )
    neuprint_fetch_stats: dict[str, object] | None = None
    flywire_fetch_stats: dict[str, object] | None = None

    try:
        parsed_flywire_materialization = parse_flywire_materialization(
            flywire_materialization
        )
        materialization_label = flywire_materialization_label(
            parsed_flywire_materialization
        )
        navis, flywire = ensure_optional_dependencies()
        client = build_neuprint_client(
            server=neuprint_server,
            dataset=neuprint_dataset,
            token_file=token_file,
        )

        print_progress(
            f"[1/4] Loading neuprint {cell_type} metadata from {neuprint_dataset}",
            enabled=True,
        )
        neuprint_meta_df = fetch_neuprint_mi1_metadata(client, cell_type)
        rng = np.random.default_rng(random_seed)
        selected_idx = int(rng.integers(len(neuprint_meta_df)))
        selected_meta = neuprint_meta_df.iloc[selected_idx].copy()
        selected_body_id = int(selected_meta["body_id"])
        print_progress(
            "  Selected neuprint body "
            f"{selected_body_id} ({selected_meta['instance']}, side={selected_meta['side']})",
            enabled=True,
        )

        print_progress(
            f"[2/4] Fetching selected neuprint skeleton {selected_body_id}",
            enabled=True,
        )
        (
            neuprint_summary_df,
            neuprint_nodes_by_body,
            neuprint_fetch_stats,
        ) = fetch_neuprint_mi1_skeleton_nodes(
            client=client,
            navis=navis,
            cell_type=cell_type,
            requested_source_space=neuprint_source_space,
            common_space=common_space,
            cache_dir=neuprint_cache_dir,
            fetch_retries=fetch_retries,
            body_ids=[selected_body_id],
            verbose=verbose,
        )
        if neuprint_summary_df.empty or selected_body_id not in neuprint_nodes_by_body:
            raise RuntimeError(f"Failed to load the selected neuprint skeleton {selected_body_id}.")
        selected_summary = neuprint_summary_df.iloc[0].copy()
        selected_nodes = neuprint_nodes_by_body[selected_body_id]

        print_progress(
            f"[3/4] Fetching FlyWire {cell_type} skeletons from "
            f"{flywire_dataset}:{parsed_flywire_materialization}",
            enabled=True,
        )
        (
            flywire_summary_df,
            flywire_nodes_by_root,
            flywire_fetch_stats,
        ) = fetch_flywire_mi1_skeleton_nodes(
            navis=navis,
            flywire=flywire,
            cell_type=cell_type,
            annotation_dataset=flywire_dataset,
            materialization=parsed_flywire_materialization,
            flywire_source_space=flywire_source_space,
            common_space=common_space,
            cache_dir=flywire_cache_dir,
            fetch_retries=fetch_retries,
            mirror_flywire=mirror_flywire,
            verbose=verbose,
        )

        print_progress(
            "[4/4] Scoring the selected neuprint skeleton against all FlyWire candidates",
            enabled=True,
        )
        candidate_scores = score_single_neuprint_skeleton_against_all_flywire(
            neuprint_row=selected_summary,
            neuprint_nodes=selected_nodes,
            flywire_summary_df=flywire_summary_df,
            flywire_nodes_by_root=flywire_nodes_by_root,
            top_k=None,
        )
        top_candidates = candidate_scores.head(int(top_k)).reset_index(drop=True)
        best_match = candidate_scores.head(1).reset_index(drop=True)
    except Exception as exc:
        if neuprint_fetch_stats is not None:
            print_fetch_summary("neuprint fetch summary", neuprint_fetch_stats)
        if flywire_fetch_stats is not None:
            print_fetch_summary("FlyWire fetch summary", flywire_fetch_stats)
        raise RuntimeError(f"Random single-neuron matching failed: {exc}") from exc

    selected_out = output_dir / f"neuprint-mi1-random-body-{selected_body_id}-summary.csv"
    top_candidates_out = output_dir / (
        f"neuprint-mi1-random-body-{selected_body_id}-to-flywire-{materialization_label}-top-{int(top_k)}.csv"
    )
    all_candidates_out = output_dir / (
        f"neuprint-mi1-random-body-{selected_body_id}-to-flywire-{materialization_label}-all-candidates.csv"
    )

    if write_outputs:
        write_csv(neuprint_summary_df, selected_out)
        write_csv(top_candidates, top_candidates_out)
        write_csv(candidate_scores, all_candidates_out)
        print(f"Wrote selected neuprint skeleton summary to {selected_out}")
        print(f"Wrote top {int(top_k)} FlyWire candidates to {top_candidates_out}")
        print(f"Wrote all FlyWire candidate scores to {all_candidates_out}")
        if not best_match.empty:
            best = best_match.iloc[0]
            print(
                "Best match: neuprint body "
                f"{int(best['neuprint_body_id'])} (side={best['neuprint_side']}) -> "
                f"FlyWire root {int(best['flywire_root_id'])} "
                f"(side={best.get('flywire_side', 'unknown')}, "
                f"mean_min_distance={best['mean_min_distance_in_common_space']:.3f})"
            )
        print_fetch_summary("neuprint fetch summary", neuprint_fetch_stats)
        print_fetch_summary("FlyWire fetch summary", flywire_fetch_stats)

    return {
        "selected_neuprint_summary": neuprint_summary_df.reset_index(drop=True),
        "selected_neuprint_nodes": {selected_body_id: selected_nodes},
        "flywire_summary": flywire_summary_df.sort_values("root_id").reset_index(drop=True),
        "flywire_nodes": flywire_nodes_by_root,
        "cache_dir": cache_dir,
        "neuprint_cache_dir": neuprint_cache_dir,
        "flywire_cache_dir": flywire_cache_dir,
        "flywire_source_space": flywire_source_space,
        "mirror_flywire": mirror_flywire,
        "best_match": best_match,
        "top_candidates": top_candidates,
        "all_candidates": candidate_scores,
        "output_dir": output_dir,
        "selected_output": selected_out,
        "top_candidates_output": top_candidates_out,
        "all_candidates_output": all_candidates_out,
        "neuprint_fetch_stats": neuprint_fetch_stats,
        "flywire_fetch_stats": flywire_fetch_stats,
    }


def main() -> int:
    args = parse_args()

    try:
        run_matching(
            cell_type=args.cell_type,
            neuprint_server=args.neuprint_server,
            neuprint_dataset=args.neuprint_dataset,
            neuprint_source_space=args.neuprint_source_space,
            flywire_dataset=args.flywire_dataset,
            flywire_materialization=args.flywire_materialization,
            flywire_source_space=args.flywire_source_space,
            mirror_flywire=args.mirror_flywire,
            common_space=args.common_space,
            output_dir=args.output_dir,
            cache_dir=args.cache_dir,
            neuprint_cache_dir=args.neuprint_cache_dir,
            flywire_cache_dir=args.flywire_cache_dir,
            token_file=args.token_file,
            fetch_retries=args.fetch_retries,
            write_outputs=True,
            verbose=args.verbose,
        )
    except Exception as exc:
        print(f"Error: {exc}", file=sys.stderr)
        return 1

    return 0


if __name__ == "__main__":
    raise SystemExit(main())


In [ ]:
from pathlib import Path
import os
import pickle
import inspect
import numpy as np

import navis
import navis.interfaces.neuprint as neu
from fafbseg import flywire

project_dir = SCRIPT_DIR  # notebook folder: skeleton caches, .neuprint_token, and match outputs live here
output_dir = project_dir / "mi1_male_cns_flywire_match"
mesh_cache_dir = output_dir / "mesh_cache"
mesh_cache_dir.mkdir(parents=True, exist_ok=True)

neuprint_body_id = 12473
flywire_root_id = 720575940618075707

NEUPRINT_SERVER = "https://neuprint.janelia.org"
NEUPRINT_DATASET = "male-cns:v0.9"
DIRECT_NEUPRINT_TOKEN = ""


def load_neuprint_token():
    if DIRECT_NEUPRINT_TOKEN.strip():
        return DIRECT_NEUPRINT_TOKEN.strip()

    for token_path in [
        project_dir / ".neuprint_token",
        Path.home() / ".neuprint_token",
    ]:
        if token_path.exists():
            return token_path.read_text(encoding="utf-8").strip()

    token = os.environ.get("NEUPRINT_APPLICATION_CREDENTIALS")
    if token:
        return token

    raise RuntimeError("neuPrint token not found.")


def call_with_supported_kwargs(func, *args, **kwargs):
    sig = inspect.signature(func)
    accepts_kwargs = any(
        p.kind == inspect.Parameter.VAR_KEYWORD
        for p in sig.parameters.values()
    )
    if accepts_kwargs:
        return func(*args, **kwargs)

    filtered = {k: v for k, v in kwargs.items() if k in sig.parameters}
    return func(*args, **filtered)


def cache_get_or_fetch(path, fetch_fn):
    if path.exists():
        with open(path, "rb") as f:
            return pickle.load(f)

    obj = fetch_fn()
    with open(path, "wb") as f:
        pickle.dump(obj, f)
    return obj


def xform_mesh_vertices(mesh, source, target, mirror_first=False):
    vertices = np.asarray(mesh.vertices, dtype=float)

    if mirror_first:
        vertices = navis.mirror_brain(vertices, template=source)

    vertices_common = navis.xform_brain(
        vertices,
        source=source,
        target=target,
    )

    return navis.MeshNeuron(
        (vertices_common, mesh.faces),
        id=mesh.id,
        name=getattr(mesh, "name", None),
        units=getattr(mesh, "units", None),
    )


def bbox_report(name, mesh):
    xyz = np.asarray(mesh.vertices, dtype=float)
    mn = xyz.min(axis=0)
    mx = xyz.max(axis=0)
    size = mx - mn
    print(name)
    print("  min:", mn)
    print("  max:", mx)
    print("  size:", size)
    print("  max_size:", size.max())
    print()


token = load_neuprint_token()
client = neu.Client(NEUPRINT_SERVER, dataset=NEUPRINT_DATASET, token=token)

flywire.set_default_dataset("public")
navis.set_pbars(jupyter=False)

neu_mesh_raw = cache_get_or_fetch(
    mesh_cache_dir / f"neuprint_mesh_{neuprint_body_id}_raw.pkl",
    lambda: call_with_supported_kwargs(
        neu.fetch_mesh_neuron,
        neuprint_body_id,
        client=client,
        lod=3,
        with_synapses=False,
    ),
)

fw_mesh_raw = cache_get_or_fetch(
    mesh_cache_dir / f"flywire_mesh_{flywire_root_id}_raw.pkl",
    lambda: flywire.get_mesh_neuron(
        flywire_root_id,
        dataset="public",
        lod=3,
        with_synapses=False,
        omit_failures=None,
        progress=True,
    ),
)

neu_mesh_common = cache_get_or_fetch(
    mesh_cache_dir / f"neuprint_mesh_{neuprint_body_id}_JRCFIB2022M.pkl",
    lambda: xform_mesh_vertices(
        neu_mesh_raw,
        source="JRCFIB2022Mraw",
        target="JRCFIB2022M",
        mirror_first=False,
    ),
)

fw_mesh_common = cache_get_or_fetch(
    mesh_cache_dir / f"flywire_mesh_{flywire_root_id}_mirrored_FLYWIRE_to_JRCFIB2022M.pkl",
    lambda: xform_mesh_vertices(
        fw_mesh_raw,
        source="FLYWIRE",
        target="JRCFIB2022M",
        mirror_first=True,
    ),
)

bbox_report("neuprint common", neu_mesh_common)
bbox_report("flywire common mirrored", fw_mesh_common)

navis.plot3d(
    [neu_mesh_common, fw_mesh_common],
    color=["tab:blue", "tab:orange"],
    width=1000,
    height=800,
)
